In [1]:
import spacy

In [2]:
spacy.__version__

'3.7.2'

In [1]:
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
import re

In [3]:

import sys
sys.path.append('../')
sys.path.append('../..')
sys.path.append('../../Output_analysis/')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

import pandas as pd

In [4]:
file = OUTPUT_DIR+'/Articles/full_text_w_genes_25_09_26_new_v.json'
genes = pd.read_json(file)


In [5]:
from Gene_modification_type import * 

In [6]:
from retrieve_product import split_sentence


In [7]:
import logging

class ColoredFormatter(logging.Formatter):
    # Define ANSI color codes
    RED = "\033[1;31m"
    GREEN = "\033[1;32m"
    BLUE = "\033[1;34m"
    PURPLE = "\033[1;35m"  # Adding purple color
    RESET = "\033[0m"

    def format(self, record):
        # Get the original log message
        message = super().format(record)

        # Apply colors based on the message content
        if "No gene-action pairs found" in message:
            return f"{self.RED}{message}{self.RESET}"
        elif "Processing article" in message:
            return f"{self.BLUE}{message}{self.RESET}"
        elif "Extracted" in message:
            return f"{self.GREEN}{message}{self.RESET}"
        elif "Possible match" in message:  # Add purple for "possible match"
            return f"{self.PURPLE}{message}{self.RESET}"
        else:
            return message  # Default (no color)

# Get the root logger
logger = logging.getLogger()

# Remove any existing handlers
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add your custom handler
handler = logging.StreamHandler()
handler.setFormatter(ColoredFormatter())
logger.addHandler(handler)

# Set the logging level (optional)
logger.setLevel(logging.INFO)

# Test the logger
logger.info("Processing article: 123")
logger.info("Extracted 5 gene-action pairs")
logger.info("No gene-action pairs found in: example_sentence")
logger.info("This is a default log message")
logger.info("Possible match: example_sentence")


Processing article: 123
Extracted 5 gene-action pairs
No gene-action pairs found in: example_sentence
This is a default log message
Possible match: example_sentence


# Search

In [8]:
def process_test_data(test, genes_mentioned, testing=False, test_sentences=[], custom_nlp=custom_nlp,
                      catch_everything=False, from_text='Title'):
    """
    Iterates through `test` DataFrame, processing abstracts and extracting information.
    """
    all_info = {}
    custom_nlp = update_gene_tagger(custom_nlp, genes_mentioned)
    genes_mentioned = make_variations_and_filter(genes_mentioned, generate_variants=False)

    for _, row in test.iterrows():

        logging.info(f"\n\n----------------------\nProcessing article:")
        #logging.info(f"{row.Title}\n")
        logging.info(f"{_}\n")
        #if testing:
        #    logging.info(f"{row.Abstract}\n")

        # --- Select source text ---
        if from_text == 'Title':
            raw_text = row.Title
        elif from_text == 'Abstract':
            raw_text = row.Abstract
        elif from_text == 'Text':
            raw_text = row.Text
        elif from_text == 'Title_Abstract':
            raw_text = row.Title + '\n' + row.Abstract
        #raw_text = row.Title + ' ' + row.Abstract

        # --- Define article genes ---
        if row.Genes and not catch_everything:
            article_genes = {gene[0] for gene in row.Genes} 
            article_genes = article_genes - {
                'ATG','ACG','CCG','CCC','ACT','AGA','CTT','CCT',
                'and','ATP','ADP','gene','genes','CoA','gene','encoding','PHB'
            }
            article_genes = make_variations_and_filter(article_genes, generate_variants=False)
            #logging.info(f"2 Clean{article_genes}\n")
        else:
            article_genes = genes_mentioned

        # --- Preprocess text into sentences ---
        sentences = preprocess_text(raw_text)
        #if testing:
        #    logging.info(f"1 Sentences {sentences}\n")

        # --- Fast substring pre-check ---
        if not genes_mentioned:    
            if not any(gene in raw_text for gene in article_genes):
                all_info[row.Title] = None
                continue
        else:
            if not any(gene in raw_text for gene in genes_mentioned):
                all_info[row.Title] = None
                continue            

        # --- Precise filtering: keep only relevant sentences ---
        relevant_sentences = get_relevant_sentences(sentences, article_genes)
        if len(relevant_sentences) == 0:
            all_info[row.Title] = None
            continue
        #logging.info(f'{relevant_sentences}')
        #print(relevant_sentences)

        # --- Clean up sentences ---
        clean_sentences = clean_up_sentences(relevant_sentences)
        #logging.info(f'{clean_sentences}')

        relevant_clean_sentences = get_relevant_sentences(clean_sentences, article_genes)
        #logging.info(f'{relevant_clean_sentences}')

        # --- Normalize sentences ---
        relevant_clean_sentences = [preprocess_sub_sentence(sentence) for sentence in relevant_clean_sentences]
        #if testing:
        #    logging.info(f"1 Clean{relevant_clean_sentences}\n")

        relevant_clean_sentences = [sentence.strip(' .,') for sentence in relevant_clean_sentences]

        # --- Expand with "i.e." matches ---
        extra_sentences = []
        for sentence in relevant_clean_sentences:
            if '(i.e.' in sentence:
                extra_sentences = re.findall(r'\(i\.e\.,\s*([^)]+)\)', sentence)
        relevant_clean_sentences += extra_sentences

        logging.info(f"Cleaned sentences: {len(relevant_clean_sentences)}")
        if len(relevant_clean_sentences) == 0:
            all_info[row.Title] = None
            continue

        #if testing:
        #    logging.info(f'{relevant_clean_sentences}')
        #    logging.info(f"2 Clean{relevant_clean_sentences}\n")

        # --- Extract gene-action relations ---
        if len(relevant_clean_sentences):       
            relevant_info = extract_genes_from_relevant_subsentences(
                relevant_clean_sentences, article_genes,
                genes_mentioned, test_sentences, custom_nlp
            )
            #print(replaced_text)

            if relevant_info:
                all_info[row.Title] = relevant_info
                #all_info.append(relevant_info)

    return all_info  #break

In [9]:
test_sentences = ['we found a mutated pcnB which resulted in decreased plasmid copy numbers and pathway enzymes to balance resource',
                 'engineering , we deleted the ROX1 gene, encoding a negative regulator of the MVA pathway and sterol biosynthesis, resulting in',
                 'the combined expression of pncB and nadE was the  effective in increasing the tolerance of the cells to  aldehydes',
                  'FkbS and accase overexpression strain SFK-OASN was ; the proincreased by 444% to 35114-units, and the FK523/ was from 96 to 56% compared with',
                  'co-overexpression of  fadD and fadAB',
                  'to further enhance production, FkbS and accase overexpression strain SFK-OASN was ; the proincreased by 444% to 35114-units, and the FK523/ was from 96 to 56% compared with that in SFK-6-33',
                  'daptomycin overproducing strain l2797-vhb, including precursor engineering (i.e.,-refactoring-kynurenine-pathway), regulatory pathway  (i.e., knocking out negative regulatory genes arpA and phaR), byproduct engineering (i.e., removing pigment), multicopy  gene cluster (BGC), and',
                  'aveC mutation increased avermectin b1a:b2a and',
                  'combined overexpression of mvk, preA, and menA increased menaquinone levels to a level',
                  'FkbS was overexpressed',
                  'enhanced by engineering aveC and precursor supply genes',
                  'pks operon was not included in the mini-cluster, but it was upregulated by SalJ activation',
                  "shewanella strains and cAMP-cyclic adenosine 3',5'-monophosphate receptor protein (CRP)  regulates multiple  EET-related pathways",
                  "gene expression of pncB and nadE respectively showed increased tolerance to furfural among",
                  "combined expression of pncB and nadE was the  effective"
                 'artificial operon containing several shikimate pathway genes, including aroE, aroB, aroF, and aroG were overexpressed to  the glucose  and',
                  'combining ELO3 and AUR1 overexpression with orm1/2δ cell viability and increased fatty acyl chain length compared',
                  'increased expression of XYL1 and XYL2 correlated with enhanced activities of XR and XDH',
                 'double deletion of murA and murB induced temperature sensitivity',
                  'hemA gene in the c4 pathway using the PA in S. meliloti 320, and the VB12 production of the engineered strain increased by 11%',
                 'combined overexpression of mvk, preA, and menA increased menaquinone levels',
                  'putative 6-phosphogluconolactonase (ykgB) gene increased uridine production by the derivative strain TD325 to 15·43-units units-1',
                  'found that deletion of degU also enhanced glucose',
                 'enhance the toward ITA production by impeding poly-β-hydroxybutyrate accumulation, which is used as and storage, via mutation of the regulator gene phaR',
                 'unexpectedly, ITA production by the phaR mutant strain',
                 'ITA production by the phaR mutant strain',
                 'mutation of the gene phaR']

In [10]:
testing=True

In [11]:
import spacy
import re
from spacy.tokens import Span
from Format_gene_modification_file import *
from text_analysis import get_variations_nlp_tagger, classify_variations
from retrieve_product import stop_words_and


re_pat = re.compile(re_pat.pattern.replace('express|',''))
re_pat = re.compile(re_pat.pattern.replace('activ|',''))
re_pat = re.compile(re_pat.pattern.replace('effect|',''))
re_pat = re.compile(re_pat.pattern.replace('increas|',''))
re_pat = re.compile(re_pat.pattern.replace('accumulat|',''))
re_pat = re.compile(re_pat.pattern.replace('engineer|','frequen|'))
re_pat = re.compile(re_pat.pattern.replace('enhanc|','additionally|'))


other_action_words = ["deletion", "removal", "disruption", "knockout", "inactivation",
                                  "introduction", "insertion", "overexpression", "mutation",
                                  "repression", "mutagenesis", "lack", "mutant", 'expression', "regulation"]

other_action_words += [word+'s' for word in other_action_words]
EXTRA = ['over', 'over-', 'under', 'under-', 're', 're-', 'co', 'co-','non','non-','down','down-','up','up-']

other_action_words += [extra_w + extend_word for extra_w in EXTRA for extend_word in other_action_words]

def log_if_testing(message, testing=False):
    if testing:
        logging.info(message)
        
#re_pat = re.compile(re_pat.pattern.replace('||','|'))
def log_dependency_table(doc):
    """
    Logs the dependency tree in a tabular format, including POS tags, lemmas, and entity information.
    """
    # Log the header
    logging.info(f"{'Text':<15} {'Lemma':<15} {'POS':<10} {'Dep':<10} {'Head':<15} {'Children':<30} {'Entity':<10}")
    logging.info("-" * 100)

    # Log each token's information
    for token in doc:
        # Get the children of the token
        children = ", ".join([f"{child._.original_text} ({child.dep_})" for child in token.children])
        # Check if the token is part of a GENE entity
        is_gene = "GENE" if token.ent_type_ == "GENE" else ""
        # Log the token's text, lemma, POS tag, dependency label, head text, children, and entity information
        logging.info(f"{token._.original_text:<15} {token.lemma_:<15} {token.pos_:<10} {token.dep_:<10} {token.head._.original_text:<15} {children:<30} {is_gene:<10}")

        
        
def extract_gene_actions_from_sub_sentence(sub_sentence, gene_names, test_sentence=False, custom_nlp=custom_nlp):
    """Extracts gene-action pairs from a sub-sentence."""
    try:
        # Update the gene_tagger with new gene_names
        #update_gene_tagger(custom_nlp, gene_names)
        # Log the gene_names used by the gene_tagger
        #if "gene_tagger" in custom_nlp.pipe_names:
            #logging.info(f"Pipe order: {custom_nlp.pipe_names}")
            #gene_tagger = custom_nlp.get_pipe("gene_tagger")
            #logging.info(f"Gene names in gene_tagger: {gene_tagger.gene_names}")
            
            
            
        doc = custom_nlp(sub_sentence)
        if test_sentence:
            log_dependency_table(doc)
            #for token in doc:
            #    logging.info(f"Token info \t{token.text}\t{token.dep_}\t{list(token.children)}\t")
        return extract_gene_actions(doc, gene_names)
    except Exception as e:
        logging.error(f"Error processing sub-sentence: {sub_sentence}. Error: {e}")
        return None

    

    
def extract_gene_actions_from_sub_sentence_with_retry(sub_sentence, gene_names, test_sentences, custom_nlp=custom_nlp):
    """
    Extracts gene-action pairs from a sub-sentence, with a retry mechanism
    if no pairs are found initially.
    """
    #logging.info(f"Test_sentence {test_sentences}")
    test_sentence = False
    org_sub = sub_sentence
    if sub_sentence:
        if sub_sentence in test_sentences:
            test_sentence = True
    else:
        logging.info(f"Subsentence empty?? {org_sub}")
    gene_actions = extract_gene_actions_from_sub_sentence(sub_sentence, gene_names, test_sentence, custom_nlp)
    
    if gene_actions:

        return sub_sentence, gene_actions
    
    new_sub_sentence = simplify_sub_sentence(sub_sentence)
    #logging.info(f"This new {new_sub_sentence} instead of {sub_sentence}")

    if (sub_sentence == 'co-overexpression of  fadD and fadAB'):
        logging.info(f"This new {new_sub_sentence}")
    if new_sub_sentence != sub_sentence:
        if new_sub_sentence in test_sentences:
            test_sentence = True
        gene_actions = extract_gene_actions_from_sub_sentence(new_sub_sentence, gene_names, test_sentence, custom_nlp)
        #logging.info(f"This new Gene{gene_actions}")
        if gene_actions:
            return new_sub_sentence, gene_actions
    
    return None, None



def get_action_verbs_and_past_actions(nlp):
    verbs = "increases improve enhance express regulate knock disrupt lack mutate repressed"  # higher
    verbs += " overexpress greater boost deletion introduction delete introduce insert remove "
    verbs += " delete inactivate deleted remove removed delete modulate modulated induce induced "
    verbs += " engineer engineered attenuation attenuate attenuated regulate regulation"

    
    variations = get_variations_nlp_tagger(verbs, nlp)
    verbs, adjectives, nouns, adverbs, ad_verb, adj_noun = classify_variations(variations)
    
    # Create a list of words and word variations to include in analysis
    words = {'overexpression', 'knock-out','overexpresse'}
    words.update(variations['NN'], variations['VBD'], variations['JJR'], variations['VBG'])
    
    words.update({extra_w + extend_word for extra_w in EXTRA for extend_word in words})
    
    action_verbs = {"delete", "knock", "inactivate", "remove", "disrupt", "inducible", "block", "regulate"}
    action_verbs.update({extra_w + extend_word for extra_w in EXTRA for extend_word in action_verbs})

    action_verbs.update(words, variations['VBP'])
    
    past_actions = {v for v in action_verbs if v.endswith('ed')}
    past_actions.update({"overexpressed", "deleted", "knocked out", "disrupted"})
    
    return action_verbs, past_actions, words, variations


# Define action verbs to track

action_verbs, past_actions, words,variations = get_action_verbs_and_past_actions(nlp)

def extract_gene_candidates(token, gene_names, remove_from_gene):
    """
    Extracts gene candidates from a token and its subtree, including conjunctions and compound dependencies.
    """
    genes = []
    if token.text.strip(remove_from_gene) in gene_names:
        genes.append(token.text)
    
    # Handle conjunctions (e.g., "HXK2 and YFG1")
    for conjunct in token.conjuncts:
        if conjunct.text.strip(remove_from_gene) in gene_names:
            genes.append(conjunct.text)
    
    # Handle compound dependencies (e.g., multi-word gene names like "fadM thioesterase")
    for child in token.children:
        if child.dep_ == "compound" and child.text.strip(remove_from_gene) in gene_names:
            genes.append(child.text)
    
    return genes

def extract_genes_from_subtree(token, gene_names, remove_from_gene):
    """
    Extracts gene names from a token's subtree, including conjunctions and the "gene" keyword.
    """
    genes = []
    for child in token.subtree:
        genes.extend(extract_gene_candidates(child, gene_names, remove_from_gene))
        
        # Handle "gene" as intermediary (e.g., "the gene HXK2")
        if child.text.lower() == "gene":
            for grandchild in child.children:
                genes.extend(extract_gene_candidates(grandchild, gene_names, remove_from_gene))
    
    return genes


def handle_negation(token, verb_form):
    """
    Adds negation handling: if the verb has a 'neg' child, mark it as negated.
    """
    for child in token.children:
        if child.dep_ == "neg":  # e.g., 'not', "n't"
            return f"not {verb_form}"
    return verb_form


def handle_phrasal_verbs(token):
    """
    Handles phrasal verbs (e.g., "activate up").
    """
    verb_form = token._.original_text.lower()
    for child in token.children:
        if child.dep_ == "prt":
            verb_form += f" {child._.original_text}"
    return verb_form

def handle_passive_voice(token):
    """
    Handles passive voice (e.g., "was activated").
    """
    if token.dep_ == "auxpass":
        return f"was {token._.original_text.lower()}"
    return token._.original_text.lower()

def construct_verb_form(token):
    """
    Constructs the verb form, including phrasal verbs and passive voice.
    """
    verb_form = handle_phrasal_verbs(token)
    verb_form = handle_passive_voice(token)
    verb_form = handle_negation(token, verb_form)

    return verb_form



def find_genes_in_dobj(child, gene_names, remove_from_gene):
    """
    Recursively finds genes inside a `dobj` (direct object) or its compounds.
    """
    found_genes = []
    for sub_child in child.children:
        if sub_child.text.strip(remove_from_gene) in gene_names:
            found_genes.append(sub_child.text)
        if sub_child.dep_ == "compound":  # Include genes in compound structures
            found_genes.extend(find_genes_in_dobj(sub_child, gene_names, remove_from_gene))
    return found_genes



def extract_passive_actions(token, gene_names, remove_from_gene):
    """
    Extracts gene-action pairs from passive voice past participles.
    """
    passive_genes = []


    for child in token.children:
        if child.dep_ == "nsubjpass":
            passive_genes.extend(extract_genes_from_subtree(child, gene_names, remove_from_gene))

            # Check for conjunctions of the passive subject
            for sibling in child.conjuncts:
                if sibling.text.strip(remove_from_gene) in gene_names:
                    passive_genes.append(sibling.text)

    # Also check if the **verb** itself has conjuncts (e.g., aroB, aroF)
    for sibling in token.conjuncts:

        # If the verb conjunct has genes as subjects, add them
        for child in sibling.children:
            if child.dep_ in {"nsubjpass", "nsubj"} and child.text.strip(remove_from_gene) in gene_names:
                passive_genes.append(child.text)

        # If the verb conjunct is a **gene itself**, add it
        if sibling.text.strip(remove_from_gene) in gene_names:
            passive_genes.append(sibling.text)

    verb_form = f"was {token._.original_text.lower()}"

    return [f"{gene.strip(remove_from_gene)} -> {verb_form}" for gene in passive_genes]


def handle_of_preposition(token, gene_names, remove_from_gene):
    """
    Handles the "of" preposition case (e.g., "deletion of HXK2").
    """
    action_genes = []
    #log_if_testing(message, testing=False)
    for obj in token.children:
        #logging.info(f'obj {obj}')
        # Direct gene mention
        gene_candidate = obj.text.strip(remove_from_gene)
        if gene_candidate in gene_names:# and gene_candidate not in action_genes:
            action_genes.append(gene_candidate)
            action_genes.extend(extract_gene_candidates(obj, gene_names, remove_from_gene))
            action_genes.extend(handle_compound_gene_names(obj, gene_names, remove_from_gene))
        
        # Handle conjunctions (e.g., "mvk, preA, and menA")
        for conjunct in obj.conjuncts:
            #logging.info(f'conjunct {conjunct}')

            if conjunct.text.strip(remove_from_gene) in gene_names:# and conjunct.text.strip(remove_from_gene) not in action_genes:
                action_genes.append(conjunct.text)
                action_genes.extend(extract_gene_candidates(conjunct, gene_names, remove_from_gene))
                action_genes.extend(handle_compound_gene_names(conjunct, gene_names, remove_from_gene))
        
        # Handle "gene" keyword (e.g., "gene HXK2")
        if obj.text.lower() == "gene":
            #logging.info(f'gene{list(obj.children)}')
    
            for grandchild in obj.children:
                action_genes.extend(extract_gene_candidates(grandchild, gene_names, remove_from_gene))
    #logging.info(f'Action_genes {action_genes}')
    return action_genes


def handle_compound_gene_names(token, gene_names, remove_from_gene):
    """
    Handles compound gene names (e.g., "fadM thioesterase gene").
    """
    action_genes = []
    if (
        any(child.text.lower().endswith("ase") for child in token.children) and  
        any(child.text.lower() == "gene" for child in token.children)
    ) and token.text != 'of':
        full_gene = f"{token.text} " + " ".join(child.text for child in token.children)
        action_genes.append(full_gene)
    return action_genes


def adjust_for_increase_decrease(token, text_to_adjust):
    """
    Adjusts the given text if it is preceded by "increase" or "decrease" modifiers.
    Ensures that "the" is included if "increase" or "decrease" is not the immediate previous word.
    
    Args:
        token: The token to check for preceding words.
        text_to_adjust: The text to adjust (e.g., token._.original_text or sibling.text).
    
    Returns:
        The adjusted text.
    """
    #logging.info(f'This is the token {token} and the text to adjust {text_to_adjust}')

    # Get up to 2 words before the token
    prev_words = token.doc[max(0, token.i - 2):token.i]
    #logging.info(f'And the previous words{prev_text}')
    #logging.info(f'checking {'induc' in prev_text}')
    
    if prev_words and (prev_words[-1].ent_type_ == "GENE" or prev_words[-1].text.startswith('gene')):
        # Go two tokens before that gene
        prev_words = token.doc[max(0, token.i - 3):token.i-1]
        #logging.info(f"Expanded prev_words due to gene entity: {[t.text for t in prev_words]}")
        if prev_words and prev_words[-1].ent_type_ == "GENE":
            prev_words = token.doc[max(0, token.i - 4):token.i-2]
            #logging.info(f"Expanded prev_words due to gene entity: {[t.text for t in prev_words]}")                   
    prev_text = " ".join([w.text for w in prev_words])

    # Check if "increase" or "decrease" is in the previous text
    if any(word in prev_text for word in ("increas", "decreas", "induc", "enhanc", "inhibit",'repress','not ')):
        # If "increase" or "decrease" is not the immediate previous word,
        # check if the next word is "the"
        #logging.info(f'Enter 1 {'induc' in prev_text}')
        #logging.info(f'Enter 1.2 {not prev_words[-1].text.lower() in ["increas", "decreas", "induc", "enhanc"]}')

        #if prev_words[-1].text.lower() not in ["increas", "decreas", "induc"]:
        #    logging.info(f'Enter 2 {'induc' in prev_text}')#

        #    if prev_words[-1].text.lower() == "the":
        #        return f'{prev_text} {text_to_adjust}'
        if prev_words[-1].text.lower() == "the":
            return f'{prev_text} {text_to_adjust}'
        elif prev_words[-1].text.lower() == 'of':
            return f'{prev_text} {text_to_adjust}'
        else:
            return f'{prev_words[-1]} {text_to_adjust}'
        
    # Check next two words
# Check next two words

    next_words = token.doc[token.i + 1: min(token.i + 3, len(token.doc))]  # Ensure we don't go out of bounds
    next_text = " ".join([w.text for w in next_words])
    #logging.info(f'And the next words: {next_text}')
    #logging.info(f'Checking if "expression" is in next text: {"expression" in next_text}')

    # Check if "expression" is in the next text (so it means it enhanced expression)
    if "expression" in next_text:
        #logging.info(f'Expression found: {"expression" in next_text}')

        # Check previous words
        if next_words and next_words[0].text.lower() == "the":
            return f'{text_to_adjust} {next_text}'
        else:
            #logging.info(f'Enter 3: {"induc" in prev_text}')

            # Ensure next_words is not empty before accessing its first element
            if next_words:
                return f'{text_to_adjust} {next_words[0].text}'

    # Return the original text if no conditions are met
    return text_to_adjust




from collections import defaultdict

def filter_modifications(results):
    gene_mod_dict = defaultdict(set)
    
    # Group modifications by gene
    for entry in results:
        gene, modification = entry.split(" -> ", 1)
        gene_mod_dict[gene].add(modification)
    
    filtered_results = []
    
    for gene, modifications in gene_mod_dict.items():
        sorted_mods = sorted(modifications, key=len, reverse=True)  # Sort by length (longest first)
        filtered_mods = []
        
        for mod in sorted_mods:
            if not any(mod in longer_mod for longer_mod in filtered_mods):
                filtered_mods.append(mod)
        
        for mod in filtered_mods:
            filtered_results.append(f"{gene} -> {mod}")
    
    return filtered_results

In [12]:
def extract_gene_actions(doc, gene_names, past_actions=past_actions, action_verbs=action_verbs):
    """
    Extracts gene-action relationships, including:
    1. Regular action verbs (e.g., "HXK2 activates YFG1")
    2. Noun-based actions (e.g., "deletion of HXK2")
    3. Past participle actions in passive voice (e.g., "HXK2 was overexpressed")
    4. Adjectival modifiers (e.g., "mutated pcnB")

    """
    results = []
    remove_from_gene = '.,*-;()'
    #logging.info(f"----------------------\nEXTRACT GENE ACTIONS")
    #logging.info(f"{doc.text}")
    relevant_tokens = [token for token in doc if len(token.text) > 3 and (token.is_alpha)]
    for token in relevant_tokens:
        #logging.info(f"\t\ttoken {token.text}")
        #logging.info(f"\t\is fake? {fake_genes(token)}")

        if fake_genes(token):
            continue
        
        token_text_lower = token.text.lower()
        if token_text_lower in other_action_words:
            #print('Caso 3')
            case3_results = extract_noun_based_actions(token, gene_names, remove_from_gene)
            results.extend(case3_results)
            #logging.info(f"Case 3: {case3_results}")
            #logging.info(f"\t {token_text_lower, token._.original_text}")
            #logging.info(f"\t ")
            
        if (token.lemma_ in action_verbs-past_actions or token_text_lower in action_verbs-past_actions) and doc[token.i-1].text not in ['was','were']\
        and token_text_lower not in variations['VBZ']:
            case1_results = extract_action_verbs(token, gene_names, remove_from_gene)
            results.extend(case1_results)
            #logging.info(f"Case 1: {case1_results}")
            #logging.info(f"\t {token.lemma_, token_text_lower, token._.original_text}")

            #logging.info(f"\t ")
            
        if token_text_lower in past_actions:
            preceding_words = {doc[i].text.lower() for i in range(max(0, token.i - 10), token.i)}
            if {"was", "were", "is", "are", "be"} & preceding_words:
                case2_results = extract_passive_actions(token, gene_names, remove_from_gene)
                results.extend(case2_results)
                #logging.info(f"Case 2: {case2_results}")
                #logging.info(f"\t {token_text_lower, token._.original_text}")

                #logging.info(f"\t ")
        

            
        if token.dep_ == "amod" and token.text.lower() in past_actions:
            if token.head.text.strip(remove_from_gene) in gene_names:
                results.append(f"{token.head.text.strip(remove_from_gene)} -> {token._.original_text.lower()}")
        #if token.text.lower() in action_verbs and token.lemma_ not in action_verbs:
        #    print('MISSING', token, token.lemma_)
    #logging.info(f'Res{results}')
    #logging.info(f'Filt{filter_modifications(results)}')

    return filter_modifications(results)

In [13]:
other = {"deletion", "removal", "disruption", "knockout",
                                  "introduction", "insertion", "overexpression",
                                  "repression", "mutagenesis", "lack", "mutant", 'expression'}
def extract_genes_from_relevant_subsentences(clean_sentences, gene_names, genes_mentioned=None,
                                            test_sentences = [], custom_nlp=custom_nlp):
    """Extracts relevant sub-sentences containing genes."""
    gene_names = gene_names - COMMON_WORDS - {'and','ATP','ADP','gene','genes'}
    #logging.info(f"Gene names: {gene_names}")

    if genes_mentioned:
        gene_names.update(genes_mentioned)
    gene_names = gene_names - COMMON_WORDS - {'and','ATP','ADP','gene','genes'}

    relevant_info = []
    for paragraph in clean_sentences:
        paragraph_words = {word.strip(REMOVE_FROM_GENE) for word in paragraph.split(' ')}-{''}
        if not gene_names.intersection(paragraph_words):
            continue  # Skip this paragraph
        
        #update_gene_tagger(custom_nlp, gene_names)

        sub_sentences = list(set(split_sentence(paragraph)+[paragraph]))  #split_sentence(paragraph) or [paragraph]
        #sub_sentences = [paragraph]
        #logging.info(f"Split_Sentence: {sub_sentences}\n")

        trim_sub_senteces = list(trim_process_sub_sentences_start_ending(sub_sentences))
                
        #logging.info(f"Trim: {trim_sub_senteces}\n")

        checked_ones = []

        for sub_sentence in trim_sub_senteces:
            sub_sentence_words = [word.strip(REMOVE_FROM_GENE) for word in sub_sentence.split(' ')]
            if gene_names.intersection(sub_sentence_words):
                #logging.info(f"Extracted21: {sub_sentence}\n")

                sub_sentence = preprocess_sub_sentence(sub_sentence)
                sub_sentence = list(trim_process_sub_sentences_start_ending([sub_sentence]))[0]
                sub_sentence = sub_sentence.strip(' ,.')
                if sub_sentence not in checked_ones:
                    #logging.info(f"Extracted123: {sub_sentence}\n")

                    result = extract_gene_actions_from_sub_sentence_with_retry(sub_sentence, gene_names, test_sentences, custom_nlp)

                    if result[0]:
                        processed_sub_sentence, gene_actions = result
                        logging.info(f"\nSub-sentence: {processed_sub_sentence}")
                        for relation in set(gene_actions):
                            logging.info(f"Extracted: {relation}\n")
                        relevant_info.append((processed_sub_sentence, gene_actions))
                    else:
                        sub_sentence_words = {word.strip(REMOVE_FROM_GENE) for word in sub_sentence.split(' ')}

                        #if sub_sentence_words.intersection(action_verbs.union(other)):
                            #log_if_testing(f"\nPossible match????", testing)


                        #logging.info(f"No gene-action pairs found in:" )
                        #logging.info(f"\t{sub_sentence}\n" )
                    checked_ones.append(sub_sentence)

                        #logging.info(f"\nGenes searching{gene_names.intersection(sub_sentence_words)}" )
                        #logging.info(f"\nActions{sub_sentence_words.intersection(action_verbs)}" )

                        #if sub_sentence_words.intersection(action_verbs.union(other)):
                        #    logging.info(f"Possible match????: {sub_sentence}\n")
    
    return relevant_info

In [14]:
import logging
import traceback
logging.basicConfig(level=logging.INFO)

def extract_noun_based_actions(token, gene_names, remove_from_gene):
    """
    Extracts gene-action pairs from noun-based actions like 'deletion of X'.
    """

    #logging.info(f"Processing token: {token.text} ({token.dep_})")
    #logging.info(f"Conjucts token: {token.conjuncts}")
    #logging.info(f"Children token: {list(token.children)}")

    action_genes = set()
    # Handle direct objects (e.g., 'ELO3' in 'combining ELO3 and AUR1 overexpression')
    if token.dep_ == "dobj" and token.text in gene_names:
        action_genes.update([token.text])
        #logging.info(f"Direct object match found: {token.text}")
    if token.dep_ == 'amod' and token.pos_ == 'ADJ':
        if token.head.text.startswith('strain'):
            for child in token.head.children:
                #print(child)
                if child.text in gene_names and child.dep_=='amod':
                    action_genes.update([child.text.strip(remove_from_gene)])


    # Handle conjunctions (e.g., 'ELO3' in 'ELO3 and AUR1 overexpression')
    for conjunct in token.conjuncts:
        if conjunct.text in gene_names and conjunct.text not in action_genes:
            action_genes.update({conjunct.text})
            #logging.info(f"Conjunct match found: {conjunct.text}")
            #logging.info(f"({action_genes}, {set(action_genes)})")
        for child in conjunct.children:
            #logging.info(f'Havingñ1{[child.text]}')
            #logging.info(f'Havingñ2{[child.text.strip(remove_from_gene)]}')

            if child.text.strip(remove_from_gene) in gene_names:
                action_genes.update([child.text.strip(remove_from_gene)])

            #logging.info(f'Havingñ{list(conjunct.children)}')
    for child in token.children:
        #logging.info(f'Havingñ{child, child.dep_}')

        # Handle "of" preposition (e.g., 'deletion of X')
        if child.dep_ == "prep" and child.text.lower() == "of":
            for for_children in child.children:
                if for_children.text == 'gene':
                    of_token_id = child.i
                    gene_token_id = for_children.i
                    if 'the' in child.doc[of_token_id+1].text:
                        # Ensure the index is within bounds before accessing
                        if gene_token_id + 1 < len(child.doc) and child.doc[gene_token_id + 1].text in gene_names:
                            action_genes.update([child.doc[gene_token_id + 1].text.strip(remove_from_gene)])
                        elif gene_token_id - 1 >= 0 and child.doc[gene_token_id - 1].text in gene_names:
                            action_genes.update([child.doc[gene_token_id - 1].text.strip(remove_from_gene)])
            
            extracted = handle_of_preposition(child, gene_names, remove_from_gene)
            #logging.info(f"Extracted from 'of' preposition: {extracted}")
            action_genes.update(extracted)
            
        # Handle compounds (e.g., 'AUR1' in 'AUR1 overexpression')
        elif child.text in gene_names and child.text not in action_genes and child.dep_ == "compound":
            action_genes.update({child.text})
            #logging.info(f"Compound match found: {child.text}")
            
        compound_extracted = handle_compound_gene_names(child, gene_names, remove_from_gene)
        #logging.info(f"Extracted from compound gene names: {compound_extracted}")
        action_genes.update(compound_extracted)

    # Remove duplicates
    #logging.info(f"({action_genes}, {set(action_genes)})")

    action_genes = list(set(action_genes))  
    #logging.info(f"Final action genes (deduplicated): {action_genes}")

    # Adjust action text for increase/decrease modifiers
    token_org_text = token._.original_text.lower()
    #logging.info(f"Original token text before adjustment: {token_org_text}")
    
    token_org_text = adjust_for_increase_decrease(token, token_org_text)
    #logging.info(f"Token text after increase/decrease adjustment: {token_org_text}")

    # Generate results
    results = [f"{gene.strip(remove_from_gene)} -> {token_org_text}" for gene in action_genes]
    #logging.info(f"Generated results before coordinated actions: {results}")

    # Handle coordinated noun-based actions
    for gene in action_genes:
        for sibling in token.conjuncts:
            sibling_text_lower = sibling.text.lower()
            #logging.info(f"Checking coordinated noun-based action: {sibling.text}")

            if sibling_text_lower in other_action_words:
                sibling_text_lower = adjust_for_increase_decrease(sibling, sibling_text_lower)
                results.append(f"{gene.strip(remove_from_gene)} -> {sibling_text_lower}")
                #logging.info(f"Added coordinated action: {gene.strip(remove_from_gene)} -> {sibling_text_lower}")

    #logging.info(f"Final extracted results: {results}")
    return results

In [15]:
def fake_genes(token):
    if token.pos_ == "VERB":
        has_agent = any(child.dep_ == "agent" for child in token.children)
        if has_agent:
            logging.info(f"Skipping '{token.text}' because it is modified by an agent (indirect action).")
            return True
        
    #if token.pos_ == "VERB" and token.dep_ == "ROOT":
    #    logging.info(f"It is a {token.pos_},{token.dep_}")
    #    logging.info(f"And has {token.lemma_},{token.head.dep_}")

        # Get subject (nsubj) of the verb
        subjects = [child for child in token.children if child.dep_ == "nsubj" or child.text == 'production']
        
        for subject in subjects:
            if subject:
                #logging.info(f"Subject found: {subject.text} ({subject.lemma_})")
                if subject.text == 'production':
                    #logging.info(f"Skipping '{token.lemma_}' because it is linked to an action noun ({subject.text}).")
                    return True
                # Skip the verb if its subject is a noun representing an action (not a gene)
                if subject.pos_ == "NOUN" and subject.lemma_ in action_verbs|{'production'}|set(other_action_words):
                    #logging.info(f"Skipping '{token.lemma_}' because it is linked to an action noun ({subject.text}).")
                    return True
    if token.text == 'of':
        return True
    return(False)

In [16]:
def extract_action_verbs(token, gene_names, remove_from_gene, past_actions=past_actions):
    """
    Extracts gene-action pairs from regular action verbs.
    """
    related_genes = []

    #logging.info(f"Processing token: {token.text} (lemma: {token.lemma_}, POS: {token.pos_})")
    

            
    # Iterate over token's children to find relevant gene-action relationships
    for child in token.children:
        subtree = False
        #logging.info(f"  Checking child: {child.text} (dep: {child.dep_})")

        # Extract genes from the token's direct dependencies
        if child.text.strip(remove_from_gene) in gene_names:
            related_genes.append(child.text)
            #logging.info(f"    Found gene in child dependency: {child.text.strip(remove_from_gene)}")
        
        # Handle children with dependencies like "dobj", "pobj", "attr", "nsubj", and "dep"
        elif child.dep_ in {"dobj", "pobj", "attr", "nsubj", "dep"}:
            extended_genes = extract_genes_from_subtree(child, gene_names, remove_from_gene)
            related_genes.extend(extended_genes)
            subtree = True
            #logging.info(f"    Found genes in subtree (dep: {child.dep_}): {extended_genes}")

        # Handle conjunctions (e.g., "murA and murB")
        if child.dep_ == "conj" and child.text.strip(remove_from_gene) in gene_names:
            related_genes.append(child.text)
            if not subtree:
                extended_genes = extract_genes_from_subtree(child, gene_names, remove_from_gene)
                related_genes.extend(extended_genes)
            #logging.info(f"    Found gene in conjunction: {child.text.strip(remove_from_gene)}")

    # Construct the verb form (e.g., 'induced', 'increasing')
    verb_form = construct_verb_form(token)
    #logging.info(f"Constructed verb form: {verb_form}")
    
    
    
    # If the verb is a clause subject or complement, check its head/root for related genes
    if token.dep_ in {"csubj", "xcomp"} and token.head:
        head = token.head
        logging.info(f"Checking head/root for clause verb '{token.text}': {head.text} ({head.dep_})")

        # Only explore if the head or its conjuncts are gene-related
        head_genes = extract_genes_from_subtree(head, gene_names, remove_from_gene)
        if head_genes:
            logging.info(f"Found additional genes from root/head ({head.text}): {head_genes}")
            related_genes.extend(head_genes)

            
    # If the verb is in past actions, filter the genes accordingly
    if verb_form in past_actions:
        #logging.info(f"Verb {verb_form} found in past actions.")
        filtered_genes = []

        # Check for direct object or passive subject dependencies
        for child in token.children:
            if child.dep_ in {"dobj", 'nsubjpass'}:
                filtered_genes.extend(find_genes_in_dobj(child, gene_names, remove_from_gene))
                #logging.info(f"    Found genes in direct object or passive subject (dep: {child.dep_}): {filtered_genes}")

            elif child.dep_ == "amod":
                if child.text.lower() in past_actions:
                    for grandchild in child.head.children:
                        if grandchild.text.strip(remove_from_gene) in gene_names:
                            filtered_genes.append(grandchild.text)
                            #logging.info(f"    Found gene in adjectival modifier: {grandchild.text.strip(remove_from_gene)}")

        # If no genes found in `dobj` structure, add them
        if filtered_genes:
            related_genes.extend(filtered_genes)
            #logging.info(f"Filtered genes after checking passive structures: {filtered_genes}")
    
    verb_form = adjust_for_increase_decrease(token, verb_form)
    #logging.info(f"Token text after increase/decrease adjustment: {verb_form}")

    # Return results in "gene -> verb" format
    result = [f"{gene.strip(remove_from_gene)} -> {verb_form}" for gene in related_genes]
    #logging.info(f"Generated action results: {result}")
    
    return result

In [17]:
def extract_genes_from_subtree(token, gene_names, remove_from_gene):
    """
    Extracts gene names from a token's subtree, including conjunctions and the "gene" keyword.
    Adds logging to verify coordination-based extraction.
    """
    genes = []
    #logging.info(f"    ▶ Token {token.text} ({token.dep_})")
    
    def get_coordinated_genes(tok):
        """Collect genes connected by conjunctions or parallel objects."""
        coord_genes = []
        for conjunct in tok.conjuncts:
            if conjunct.text.strip(remove_from_gene) in gene_names:
                coord_genes.append(conjunct.text.strip(remove_from_gene))
                #logging.info(f"      Found coordinated gene via conjunct: {conjunct.text.strip(remove_from_gene)}")

        if tok.dep_ in {"dobj", "pobj", "attr", "nmod"} and tok.head:
            for sibling in tok.head.children:
                if sibling.dep_ == tok.dep_ and sibling.text.strip(remove_from_gene) in gene_names:
                    coord_genes.append(sibling.text.strip(remove_from_gene))
                    logging.info(f"      Found coordinated gene via sibling: {sibling.text.strip(remove_from_gene)}")
        return list(set(coord_genes))

    #logging.info(f"    ▶ Extracting genes from subtree of token: {token.text} ({token.dep_})")

    for child in token.subtree:
        #logging.info(f"      Inspecting child in subtree: {child.text} ({child.dep_})")

        # Direct gene detection
        if child.text.strip(remove_from_gene) in gene_names:
            genes.append(child.text.strip(remove_from_gene))
            #logging.info(f"        ✓ Found gene: {child.text.strip(remove_from_gene)}")

            # Check coordination directly on that gene
            coordinated = get_coordinated_genes(child)
            if coordinated:
                #logging.info(f"        ↪ Coordinated genes detected for {child.text}: {coordinated}")
                genes.extend(coordinated)

        # Handle phrases like "the gene HXK2"
        if child.text.lower() == "gene":
            for grandchild in child.children:
                if grandchild.text.strip(remove_from_gene) in gene_names:
                    genes.append(grandchild.text.strip(remove_from_gene))
                    #logging.info(f"        ✓ Found gene through 'gene' keyword: {grandchild.text.strip(remove_from_gene)}")

    unique_genes = list(set(genes))
    #logging.info(f"    ✅ Final genes extracted from subtree of '{token.text}': {unique_genes}")
    return unique_genes

In [18]:

def preprocess_sub_sentence(sub_sentence):
    """Preprocesses a sub-sentence by replacing specific phrases with regex to handle word boundaries."""
    replacements = {
        r'\bthe sigma factor\b': 'the',
        r'\bcoding gene\b': 'gene',
        r'\band another\b': 'and',
        r'\bmutation in\b': 'mutated',
        r'(\bto further\b\s)?\benhance production,': ',',
        r'\bhave met with\b': 'experienced',
        r'\bgenes causing increased ring formation\b': 'genes',
        r'\bthe gene cluster regulator\b':'',
        r'(\bthe )?(relative )?\btranscription(al)? level(s)? of\b':'the expression of',

#        r'\bthe transcription levels of\b':'the expression of',

        r'\bover-':'over',
        r'\bdown-':'down',
        r'\bmutation of the regulator gene(s)?\b': 'mutation of the gene',
        r'\bintroducing\s\w+\samino\s?acid\smutations\b':'mutating',
        r'\bintroduc\w+\sa\smutations?\b':'mutating'
#        r'\bthe toward\b':'the',

        
    }
    
    for old, new in replacements.items():
        sub_sentence = re.sub(old, new, sub_sentence)
    
    return sub_sentence

------
# Finish test

In [19]:
correct = [{'ldhA -> deleting'},
 {'phaC -> enhance the expression'},
 {'pcnB -> mutated'},
 {'murA -> deletion', 'murB -> deletion'},
 {'ROX1 -> deleted', 'ERG13 -> overexpressed'},
 {'HXK2 -> deletion', 'HXK2 -> deleting'},
 {'hemA -> overexpressed'},
 {'nadE -> expression', 'pncB -> expression'},
 {'FkbS -> was overexpressed'},
 {'menA -> overexpression', 'preA -> overexpression', 'mvk -> overexpression'},
 {'fadD -> was overexpressed',
  'aveC -> engineering','aveC -> mutation', 'fadD -> overexpression'},
 {'phaR -> knocking', 'arpA -> knocking'},
 {'aroB -> was overexpressed',
  'aroF -> was overexpressed',
  'aroG -> was overexpressed',
  'aroK -> was disrupted'},
 {'ORM1 -> deletion', 'AUR1 -> overexpression', 'ELO3 -> overexpression'},
 {'ansB -> increased the expression', 'ykgB -> overexpression', 'aprE -> inducing expression', 'ansB -> overexpression'} ,
 {'XKS1 -> increasing the expression',
  'XYL1 -> increased expression',
  'XYL1 -> increasing the expression',
  'XYL2 -> increased expression',
  'XYL2 -> increasing the expression',
  'ZWF1 -> increased'},
 {'degU -> deletion'}]

In [20]:
#'ykgB -> increased', Checar 

# First

In [21]:


genes_mentioned = set()
for _, row in genes.dropna(subset=['Genes']).iterrows():
        #if testing:
            
        #    logging.info(f"{row.Abstract}\n")

        #raw_text = row.Title + ' ' + row.Abstract
        
        genes_mentioned.update([gene[0] for gene in row.Genes])


In [22]:
genes_mentioned = genes_mentioned | {'nth','nlp','crt'}

In [23]:
import itertools
import string
import re

# --- Constants ---
ROMANS = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
UPPERCASE = string.ascii_uppercase
DIGITS = [str(i) for i in range(10)]
ALL_CODONS = {''.join(codon) for codon in itertools.product("ATCGU", repeat=3)}

# --- Gene Variant Generation ---
def generate_gene_variants(genes_mentioned):
    """
    Generate potential variants for each gene based on suffix patterns:
    - Numeric endings (e.g., 'gene1' → gene0–gene9, geneI–geneX)
    - Uppercase letter endings (e.g., 'geneA' → geneA–geneZ, geneA0–9)
    - Roman numerals (e.g., 'geneIV' → geneIV–geneX)
    - Lowercase genes (adds A–Z and 0–9 variants)
    """
    gene_variants = set()

    for gene in genes_mentioned:
        variants = {gene}
        base_gene = gene

        # --- Numeric suffix ---
        if re.search(r"\d+$", gene):
            base_gene = re.sub(r"\d+$", "", gene)
            variants |= {f"{base_gene}{i}" for i in DIGITS}
            variants |= {f"{base_gene}{r}" for r in ROMANS}

        # --- Uppercase letter suffix ---
        elif re.search(r"[A-Z]$", gene):
            base_gene = gene[:-1]
            variants |= {f"{base_gene}{letter}" for letter in UPPERCASE}
            variants |= {f"{base_gene}{i}" for i in DIGITS}
            variants |= {f"{gene}{i}" for i in DIGITS}
            variants |= {f"{gene}{r}" for r in ROMANS}

        # --- Lowercase gene ---
        if gene.islower():
            variants |= {f"{gene}{i}" for i in DIGITS}
            variants |= {f"{gene}{letter}" for letter in UPPERCASE}

        # --- Roman numeral suffix ---
        for r in ROMANS:
            if gene.endswith(r):
                base_gene = gene[:-len(r)]
                start_index = ROMANS.index(r)
                variants |= {f"{base_gene}{n}" for n in ROMANS[start_index:]}
                break

        gene_variants |= variants

    return gene_variants


def make_variations_and_filter(genes_mentioned, generate_variants=True):
    """
    Cleans and optionally expands gene names.
    
    Steps:
      - Removes common words, codons, and non-gene terms.
      - Optionally generates and filters variants.
      - Removes single-letter + single-digit patterns like 'c1', 'a2', etc.
    """
    genes = set(genes_mentioned)

    # --- Step 1: Base filtering ---
    filtered = genes - COMMON_WORDS - ALL_CODONS

    # Remove known non-gene or ambiguous terms
    non_genes = {
        'CoA', 'gene', 'encoding', 'ATCC', 'PCC', 'CHO', 'III', 'CO2', 'CO',
        'PHA', 'PHB', 'ABC', 'c3', 'c4', 'c5', 'GTP', 'IPTG', 'pH','NO','kan',
        'and', 'ATP', 'ADP', 'genes', 'PCR', 'FAD', 'see', 'cas9','CA','UDP',
        'log','cas', 'CDW', 'DW','hot', 'NH','SO','rpm','affinity'
    }
    filtered -= non_genes
    filtered -= set(ROMANS)
    filtered -= {r.lower() for r in ROMANS}

    # --- Step 2: Remove short ambiguous names like 'c1', 'a2', etc. ---
    filtered = {g for g in filtered if not re.fullmatch(r"[a-zA-Z]\d{1,2}$", g)}

    # --- Step 3: Generate variants (optional) ---
    if generate_variants:
        variants = generate_gene_variants(filtered)
        variants -= COMMON_WORDS | ALL_CODONS | non_genes
        variants -= set(ROMANS) | {r.lower() for r in ROMANS}
        variants = {g for g in variants if not re.fullmatch(r"[a-zA-Z]\d$", g)}
        filtered |= variants

    return filtered

genes_mentioned.update(set(['levD']))

In [24]:
gene_variants = make_variations_and_filter(genes_mentioned)

In [25]:
#maybe not 24113893
genes = genes.loc[~genes.index.isin([39273688,24113893])]

In [26]:
check_genes = set(['ado','cgm','hdeAB','dhaD','glck','amyB','agdA','psy','lcy','crtE','OMT',
             'dbv20','frr','purR','glnR','pnox','FNS','sps','sps2','galU','ptsG',
            'npht7','HmCA','ubi','ubiC','hdeAB','cgm','dhaD','dxs','dxt','mgs'])
for n in gene_variants:
    if n in check_genes:
        check_genes.remove(n)

        
gene_variants.update(check_genes)

In [27]:
gene_variants = make_variations_and_filter(gene_variants)

In [28]:
#hdeAB
#cgm
#dhaD

In [29]:
28610971

28610971

In [30]:
15116434

15116434

In [31]:
import logging

# Remove any previous logging handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging to file only
logging.basicConfig(
    filename='logfile2.log',        # path to log file
    filemode='w',                  # overwrite each run (use 'a' to append)
    level=logging.INFO,            # or DEBUG, WARNING, etc.
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Optional: verify it works
logging.info("Logging is now redirected to logfile.log")


In [32]:
#check the problem
#results = process_test_data(genes.dropna(subset=['Genes']),genes_mentioned = gene_variants)
results = process_test_data(genes,genes_mentioned = gene_variants, from_text='Title_Abstract')

In [33]:
articulos = pd.DataFrame([results]).T.dropna().index

In [34]:
genes_title = genes.loc[genes.Title.isin(articulos)]

In [35]:
genes_faltan = genes.loc[~genes.Title.isin(articulos)]

In [36]:
for n in gene_variants:
    if n in ['FPS1','ADH2']:
        print(n)

ADH2
FPS1


In [37]:
#32209089 check
#32710115 check
#33957462 check

In [38]:
#ATCC IS NOT A GENE PCC, CHO, III, CO2, PHA, PHB, ABC
#PCA? GTP, c3, c4, c5
#MVA?
#fat?
#cas9

In [39]:
len(results)

12762

In [40]:
pd.DataFrame([results]).T.to_pickle('Datos_largo_test_3_7_oct.pickle')

In [41]:
results
paper_modifications = {}
for n,paper in results.items():
    
    modification = set()
    if paper:
        for sentence in paper:
            if sentence:
                modification.update(sentence[1])
        paper_modifications[n] = modification
    
paper_modifications

{'Increased production of zeaxanthin and other pigments by application of genetic engineering techniques to Synechocystis sp. strain PCC 6803.': {'crtB -> expression',
  'crtB -> overexpression',
  'crtB -> was introduced',
  'crtP -> expression',
  'crtP -> overexpression',
  'crtP -> was introduced',
  'crtR -> expression',
  'crtR -> was introduced',
  'ipi -> expression',
  'ipi -> was introduced'},
 'Altered regulation of pyruvate kinase or co-overexpression of phosphofructokinase increases glycolytic fluxes in resting Escherichia coli.': {'pyk -> overexpression'},
 'A novel genetically engineered pathway for synthesis of poly(hydroxyalkanoic acids) in Escherichia coli.': {'buk -> expressing',
  'phaC -> expressing',
  'phaE -> expressing',
  'ptb -> expressing'},
 'Metabolic engineering of carotenoid accumulation in Escherichia coli by modulation of the isoprenoid precursor pool with expression of deoxyxylulose phosphate synthase.': {'DXS -> not expressing',
  'dxs -> overexpress

In [42]:
len(paper_modifications)

3425

In [43]:
3283

3283

## Second round

In [44]:
table = pd.DataFrame([paper_modifications]).T
table.columns = ['Genes']
genes_mentioned = set(table.explode('Genes').Genes.str.split(' ').str[0].to_list())

In [45]:
genes_mentioned = genes_mentioned - {'encoding','gene'}
genes_mentioned = genes_mentioned | {'nth','nlp','crt','aca','atf','ICDH'}

In [46]:
gene_variants_2 = make_variations_and_filter(genes_mentioned)

In [47]:
all_gene_variants = gene_variants_2.union(gene_variants)


In [48]:
for n in all_gene_variants:
    if n=='ICDH':
        print(n)

ICDH


In [49]:
#check the problem
#results2 = process_test_data(genes.dropna(subset=['Genes']),genes_mentioned = all_gene_variants)
results2 = process_test_data(genes,genes_mentioned = all_gene_variants,
                            catch_everything=True, from_text='Abstract')

CHECK WHICH ARE THE NEW GENES AND 

In [ ]:
results2

In [51]:
len(results2)

11658

In [52]:
results2
paper_modifications_2 = {}
for n,paper in results2.items():
    
    modification = set()
    if paper:
        for sentence in paper:
            if sentence:
                modification.update(sentence[1])
        paper_modifications_2[n] = modification
    
paper_modifications_2

{'Increased production of zeaxanthin and other pigments by application of genetic engineering techniques to Synechocystis sp. strain PCC 6803.': {'crtB -> expression',
  'crtB -> overexpression',
  'crtB -> was introduced',
  'crtP -> expression',
  'crtP -> overexpression',
  'crtP -> was introduced',
  'crtR -> expression',
  'crtR -> overexpression',
  'crtR -> was introduced',
  'ipi -> expression',
  'ipi -> overexpression',
  'ipi -> was introduced'},
 'Altered regulation of pyruvate kinase or co-overexpression of phosphofructokinase increases glycolytic fluxes in resting Escherichia coli.': {'pyk -> overexpression'},
 'A novel genetically engineered pathway for synthesis of poly(hydroxyalkanoic acids) in Escherichia coli.': {'buk -> expressing',
  'phaC -> expressing',
  'phaE -> expressing',
  'ptb -> expressing'},
 'Properties of engineered poly-3-hydroxyalkanoates produced in recombinant Escherichia coli strains.': {'PhbB -> introducing',
  'phbB -> introduction'},
 'Metaboli

In [53]:
len(paper_modifications_2)

4227

In [54]:
pd.DataFrame([results2]).T.to_pickle('Datos_largo_test_3_7_oct_2.pickle')

# All papers

In [55]:
all_first_batch = genes.dropna(subset=['Genes','Text'])
df_split = np.array_split(all_first_batch, 10)

In [56]:
all_first_batch = genes.dropna(subset=['Genes','Text'])


In [57]:
already_retrieved= pd.DataFrame([paper_modifications_2]).T.dropna().index

In [58]:
all_first_batch.loc[~all_first_batch.Title.isin(already_retrieved)].Gene_Source.value_counts()

Gene_Source
full_text    4180
abstract      726
title           1
Name: count, dtype: int64

In [59]:
all_first_batch_filtered = all_first_batch.loc[~all_first_batch.Title.isin(already_retrieved)]
df_split = np.array_split(all_first_batch_filtered, 10)

In [60]:
batch = 0
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [61]:
batch = 1
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [62]:
batch = 2
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [63]:
batch = 3
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [64]:
batch = 4
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [65]:
batch = 5
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [66]:
batch = 6
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [67]:
batch = 7
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [68]:
batch = 8
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [69]:
batch = 9
results_full = process_test_data(df_split[batch], genes_mentioned=all_gene_variants,
                                catch_everything=True, from_text='Text')
pd.DataFrame([results_full]).T.to_pickle(f'3Datos_largo_7_oct_pt{batch+1}.pickle')

In [ ]:
all_gene_variants = make_variations_and_filter(all_gene_variants, generate_variants=False)

In [ ]:
for n in all_gene_variants:
    if n in ['see','pH','b12','levD']:
        print(n)

why is it extracting of?

In [70]:
data_frames = []
for n in range(0,10):
    data_part = pd.read_pickle(f'3Datos_largo_7_oct_pt{n+1}.pickle')    
    data_frames.append(data_part)
all_files = pd.concat(data_frames)

In [75]:
paper[0][0]

('of PKS engineering strategies from domain and module substitutions to subunit - tation and PKS fusions can be used for the production levels of polyketides',
 ['PKS -> engineering'])

In [77]:
all_paper_modifications = {}
for n,paper in all_files.iterrows():

    modification = set()
    if paper[0]:
        for sentence in paper[0]:
            modification.update(sentence[1])
        all_paper_modifications[n] = modification
    
all_paper_modifications

{'Formation of functional heterologous complexes using subunits from the picromycin, erythromycin and oleandomycin polyketide synthases.': {'PKS -> engineering',
  'PKS -> express',
  'PKS -> expression',
  'pikAIII -> removed',
  'pikAIV -> removed'},
 'Engineering desiccation tolerance in Escherichia coli.': {'PAO -> increased',
  'spsA -> expression',
  'spsA -> overexpression',
  'spsA -> remove',
  'sur -> increased',
  'sur -> was enhanced'},
 'Cloning, nucleotide sequence, and heterologous expression of the biosynthetic gene cluster for R1128, a non-steroidal estrogen receptor antagonist. Insights into an unusual priming mechanism.': {'CLF -> lacking',
  'MAT -> lacking',
  'SCP2 -> insert'},
 'The biosynthetic gene cluster for the antitumor drug bleomycin from Streptomyces verticillus ATCC15003 supporting functional interactions between nonribosomal peptide synthetases and a polyketide synthase.': {'PCP -> regulation',
  'pbs12 -> introduction',
  'pbs12 -> was introduced',
  '

-----

In [ ]:
import pandas as pd
import numpy as np

def safe_union(set1, set2):
    # Replace NaN with an empty set
    if pd.isna(set1):
        set1 = set()
    if pd.isna(set2):
        set2 = set()
    return set1.union(set2)

# Assuming your DataFrame is named all_modifications:
all_modifications["Union"] = all_modifications.apply(
    lambda row: safe_union(row["Title_Abstract"], row["Full_text"]), axis=1
)

# Display the DataFrame
all_modifications

In [ ]:
all_modifications.to_csv('modificaciones')

In [ ]:
all_modifications.Union.explode().str.split(' -> ').str[0].value_counts()